In [5]:
import pandas as pd

In [6]:
xlxs = pd.ExcelFile('/home/klby/Documents/DataScrapeV2/CollegeScorecardDataDictionary.xlsx')
xlxs.sheet_names

['README',
 'ChangeLog',
 'Glossary',
 'Institution_Data_Dictionary',
 'Institution_Cohort_Map',
 'Most_Recent_Inst_Cohort_Map',
 'FieldOfStudy_Data_Dictionary',
 'FieldOfStudy_Cohort_Map']

In [7]:
df1 = pd.DataFrame
df2 = pd.DataFrame
df3 = pd.DataFrame
df4 = pd.DataFrame
df5 = pd.DataFrame
df6 = pd.DataFrame
for i in range(1, 7):
    globals()[f'df{i-1}'] = xlxs.parse(xlxs.sheet_names[i])

In [14]:
restoration_sql = """
-- =====================================================================
-- PRESERVED NORMALIZED TABLES
-- =====================================================================

CREATE TABLE IF NOT EXISTS institution_demographics (
    id INT PRIMARY KEY AUTO_INCREMENT,
    institution_id INT NOT NULL,
    race_ethnicity_id INT NOT NULL,
    percentage DECIMAL(5, 2),
    category VARCHAR(50),
    cohort_year_id INT,
    data_quality_flag VARCHAR(50),
    FOREIGN KEY (institution_id) REFERENCES institution(id) ON DELETE CASCADE,
    FOREIGN KEY (race_ethnicity_id) REFERENCES race_ethnicity(id),
    FOREIGN KEY (cohort_year_id) REFERENCES cohort_year(id),
    UNIQUE KEY unique_institution_race_cohort (institution_id, race_ethnicity_id, cohort_year_id, category),
    INDEX idx_institution_id (institution_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS field_of_study (
    id INT PRIMARY KEY AUTO_INCREMENT,
    cip_code VARCHAR(10) NOT NULL,
    cip_description VARCHAR(255) NOT NULL,
    credential_level_id INT NOT NULL,
    FOREIGN KEY (credential_level_id) REFERENCES credential_level(id),
    UNIQUE KEY unique_cip_credential (cip_code, credential_level_id),
    INDEX idx_cip_code (cip_code),
    INDEX idx_credential_level_id (credential_level_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS institution_field_of_study (
    id INT PRIMARY KEY AUTO_INCREMENT,
    institution_id INT NOT NULL,
    field_of_study_id INT NOT NULL,
    cohort_year_id INT,
    award_count INT,
    FOREIGN KEY (institution_id) REFERENCES institution(id) ON DELETE CASCADE,
    FOREIGN KEY (field_of_study_id) REFERENCES field_of_study(id) ON DELETE CASCADE,
    FOREIGN KEY (cohort_year_id) REFERENCES cohort_year(id),
    UNIQUE KEY unique_institution_field_cohort (institution_id, field_of_study_id, cohort_year_id),
    INDEX idx_institution_id (institution_id),
    INDEX idx_field_of_study_id (field_of_study_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS field_of_study_earnings (
    id INT PRIMARY KEY AUTO_INCREMENT,
    institution_field_of_study_id INT NOT NULL,
    cohort_year_id INT NOT NULL,
    median_earnings_5yr INT,
    monthly_earnings_5yr INT,
    data_quality_flag VARCHAR(50),
    measurement_year INT,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    FOREIGN KEY (institution_field_of_study_id) REFERENCES institution_field_of_study(id) ON DELETE CASCADE,
    FOREIGN KEY (cohort_year_id) REFERENCES cohort_year(id),
    INDEX idx_institution_field_of_study_id (institution_field_of_study_id),
    INDEX idx_cohort_year_id (cohort_year_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS field_of_study_debt (
    id INT PRIMARY KEY AUTO_INCREMENT,
    institution_field_of_study_id INT NOT NULL,
    cohort_year_id INT NOT NULL,
    median_debt_all_schools DECIMAL(10, 2),
    median_debt_this_school DECIMAL(10, 2),
    monthly_payment_all_schools_10yr INT,
    monthly_payment_this_school_10yr INT,
    data_quality_flag VARCHAR(50),
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    FOREIGN KEY (institution_field_of_study_id) REFERENCES institution_field_of_study(id) ON DELETE CASCADE,
    FOREIGN KEY (cohort_year_id) REFERENCES cohort_year(id),
    INDEX idx_institution_field_of_study_id (institution_field_of_study_id),
    INDEX idx_cohort_year_id (cohort_year_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

CREATE TABLE IF NOT EXISTS api_metadata (
    id INT PRIMARY KEY AUTO_INCREMENT,
    institution_id INT NOT NULL UNIQUE,
    api_etag VARCHAR(255),
    last_fetched_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (institution_id) REFERENCES institution(id) ON DELETE CASCADE,
    INDEX idx_institution_id (institution_id)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;

-- =====================================================================
-- REFERENCE DATA (Sample)
-- =====================================================================
INSERT IGNORE INTO credential_level (credential_code, credential_name, description) VALUES
(1, 'Undergraduate Certificates or Diplomas', 'Sub-baccalaureate certificate or diploma programs'),
(2, 'Associate\\'s Degrees', '2-year associate degree programs'),
(3, 'Bachelor\\'s Degrees', '4-year bachelor degree programs'),
(4, 'Post-baccalaureate Certificates', 'Certificates for those with bachelor\\'s degrees'),
(5, 'Master\\'s Degrees', 'Master\\'s degree programs'),
(6, 'Doctoral Degrees', 'Research and professional doctoral degrees'),
(7, 'First Professional Degrees', 'First professional degrees (Law, Medicine, etc)'),
(8, 'Graduate/Professional Certificates', 'Graduate and professional certificates');

INSERT IGNORE INTO race_ethnicity (race_code, race_name, description) VALUES
('WHITE', 'White', 'Non-Hispanic White'),
('BLACK', 'Black', 'Non-Hispanic Black'),
('HISP', 'Hispanic', 'Hispanic (any race)'),
('ASIAN', 'Asian', 'Non-Hispanic Asian'),
('AIAN', 'American Indian/Alaska Native', 'American Indian/Alaska Native'),
('NHPI', 'Native Hawaiian/Pacific Islander', 'Native Hawaiian/Pacific Islander'),
('2MOR', 'Two or More Races', 'Non-Hispanic Two or More Races'),
('NRA', 'Non-Resident Alien', 'Non-Resident Alien'),
('UNKN', 'Unknown', 'Race/ethnicity unknown');

INSERT IGNORE INTO region (region_code, region_name, description) VALUES
(0, 'New England', 'CT, ME, MA, NH, RI, VT'),
(1, 'Mid East', 'DE, DC, MD, NJ, NY, PA'),
(2, 'Great Lakes', 'IL, IN, MI, OH, WI'),
(3, 'Plains', 'IA, KS, MN, MO, NE, ND, SD'),
(4, 'Southeast', 'AL, AR, FL, GA, KY, LA, MS, NC, SC, TN, VA, WV'),
(5, 'Southwest', 'OK, TX'),
(6, 'Rocky Mountains', 'CO, ID, MT, UT, WY'),
(7, 'Far West', 'AK, AZ, CA, HI, NV, NM, OR, WA'),
(8, 'Outlying Areas', 'AS, FM, GU, MH, MP, PR, PW, VI');

INSERT IGNORE INTO institution_type (type_code, type_name, description) VALUES
(1, 'Public', 'Publicly funded institution'),
(2, 'Private Nonprofit', 'Private nonprofit institution'),
(3, 'Private For-Profit', 'Private for-profit institution');

INSERT IGNORE INTO degree_type (degree_code, degree_name, description) VALUES
(1, 'Certificate Degree', 'Certificate-granting institution'),
(2, 'Associate Degree', 'Associate degree-granting institution'),
(3, 'Bachelor Degree', 'Bachelor degree-granting institution'),
(4, 'Graduate Degree', 'Graduate degree-granting institution');
"""

with open('/home/klby/Documents/DataScrapeV2/schema.sql', 'a') as f:
    f.write(restoration_sql)

print("Restored missing tables and reference data.")

Restored missing tables and reference data.
